In [ ]:
# Imports et configuration
import sys
from pathlib import Path

# Ajouter le chemin src au path si nécessaire
project_root = Path.cwd().parent
if str(project_root / 'src') not in sys.path:
    sys.path.insert(0, str(project_root / 'src'))

from pyvolley.scrapers import FFVBScraper
from pyvolley.core.config import settings

print(f"Base URL: {settings.ffvb_base_url}")
print(f"PDFs dir: {settings.pdfs_dir}")

In [ ]:
# Initialisation du scraper
scraper = FFVBScraper()
print(f"Scraper: {scraper.name}")
print(f"Base URL: {scraper.base_url}")

In [ ]:
# Test 1: Récupérer la liste des ligues
print("Récupération des ligues...")
ligues = scraper.get_ligues()
print(f"Nombre de ligues trouvées: {len(ligues)}")

# Afficher les premières ligues
for ligue in ligues[:10]:
    print(f"  - {ligue['code']}: {ligue['nom']} (session: {ligue.get('session', 'N/A')})")

In [ ]:
# Afficher toutes les ligues disponibles
print(f"\nToutes les ligues ({len(ligues)} trouvées):")
for ligue in ligues:
    print(f"  - {ligue['code']:10} | {ligue['nom'][:50]:50} | session: {ligue.get('session', 'N/A')}")

In [ ]:
# Test 2: Récupérer les compétitions d'une ligue (choisir une ligue de test)
if ligues:
    test_ligue = ligues[0]
    print(f"\nTest avec la ligue: {test_ligue['code']} - {test_ligue['nom']}")
    print(f"Session: {test_ligue.get('session', 'N/A')}")
    
    competitions = scraper.get_competitions(
        ligue_code=test_ligue['code'], 
        saison=test_ligue.get('session')
    )
    print(f"\nNombre de compétitions trouvées: {len(competitions)}")
    
    for comp in competitions[:20]:
        print(f"  - {comp.code}: {comp.nom} (genre: {comp.genre}, cat: {comp.categorie})")

In [ ]:
# Test 3: Récupérer les poules d'une compétition
if competitions:
    test_comp = competitions[0]
    print(f"\nTest avec la compétition: {test_comp.code} - {test_comp.nom}")
    
    poules = scraper.get_poules(test_comp.code, test_comp.ligue_code)
    print(f"Nombre de poules trouvées: {len(poules)}")
    
    for poule in poules[:10]:
        print(f"  - {poule['code']}: {poule['nom']}")

In [ ]:
# Test 4: Récupérer les matchs d'une compétition
if competitions:
    test_comp = competitions[0]
    print(f"\nRécupération des matchs pour: {test_comp.code} - {test_comp.nom}")
    
    matches = list(scraper.get_matches(test_comp))
    print(f"Nombre de matchs trouvés: {len(matches)}")
    
    for match in matches[:10]:
        print(f"  - {match.code}: PDF URL = {match.pdf_url}")

In [ ]:
# Test 5: Télécharger un PDF de test
if matches:
    test_match = matches[0]
    output_dir = project_root / "data" / "pdfs" / "test"
    
    print(f"\nTéléchargement de: {test_match.code}")
    print(f"URL: {test_match.pdf_url}")
    print(f"Destination: {output_dir}")
    
    result = scraper.download_match_pdf(test_match, output_dir)
    print(f"\nRésultat: {result.success}")
    print(f"Message: {result.message}")
    if result.data:
        print(f"Taille: {result.data.get('size', 0)} octets")

# Analyse des Ligues et Comités

Testons comment récupérer les poules pour les ligues (ex: LIRA) et comités.

In [ ]:
# Analyser la structure de la page calendrier pour LIRA
import requests
from bs4 import BeautifulSoup
from urllib.parse import urljoin, urlencode

base_url = "https://www.ffvbbeach.org/ffvbapp/resu/"
entity_code = "LIRA"  # Ligue Auvergne-Rhône-Alpes
saison = "2025/2026"

params = {
    "saison": saison,
    "codent": entity_code,
}
url = urljoin(base_url, f"vbspo_calendrier.php?{urlencode(params)}")
print(f"URL: {url}")

response = requests.get(url)
soup = BeautifulSoup(response.text, "html.parser")

# Afficher les 500 premiers caractères pour voir la structure
print(f"\nStatus: {response.status_code}")
print(f"\nContenu (500 chars):\n{response.text[:500]}")